In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [7]:
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

In [8]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

In [9]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self,x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)
model = MNISTClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device: cuda


In [11]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.11.0+cu128
12.8
True


In [12]:
def train_epoch(model, loss_function, optimizer, train_loader, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = loss_function(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

        if batch_idx % 100 == 0 and batch_idx > 0:
            avg_loss = running_loss / 100
            accuracy = 100.*correct/total
            print(f' [{batch_idx * 64}/60000]'
                  f' Loss: {avg_loss:.3f} | Accuracy: {accuracy:.3f}')
            running_loss = 0.0

In [13]:
def evaluate(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            output = model(inputs)
            _, predicted = output.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    return 100. *correct/total

In [14]:
num_epochs = 10
for epoch in range(num_epochs):
    print('epoch:', epoch)
    train_epoch(model, loss_function, optimizer, train_loader, device)
    accuracy = evaluate(model, test_loader, device)
    print(f'accuracy: {accuracy:.3f}')

epoch: 0
 [6400/60000] Loss: 0.639 | Accuracy: 81.235
 [12800/60000] Loss: 0.324 | Accuracy: 85.627
 [19200/60000] Loss: 0.256 | Accuracy: 87.843
 [25600/60000] Loss: 0.235 | Accuracy: 89.109
 [32000/60000] Loss: 0.199 | Accuracy: 90.188
 [38400/60000] Loss: 0.191 | Accuracy: 90.929
 [44800/60000] Loss: 0.181 | Accuracy: 91.499
 [51200/60000] Loss: 0.169 | Accuracy: 91.957
 [57600/60000] Loss: 0.142 | Accuracy: 92.375
accuracy: 95.830
epoch: 1
 [6400/60000] Loss: 0.122 | Accuracy: 96.380
 [12800/60000] Loss: 0.121 | Accuracy: 96.525
 [19200/60000] Loss: 0.122 | Accuracy: 96.434
 [25600/60000] Loss: 0.114 | Accuracy: 96.489
 [32000/60000] Loss: 0.108 | Accuracy: 96.551
 [38400/60000] Loss: 0.117 | Accuracy: 96.542
 [44800/60000] Loss: 0.110 | Accuracy: 96.558
 [51200/60000] Loss: 0.103 | Accuracy: 96.586
 [57600/60000] Loss: 0.113 | Accuracy: 96.598
accuracy: 96.990
epoch: 2
 [6400/60000] Loss: 0.079 | Accuracy: 97.509
 [12800/60000] Loss: 0.073 | Accuracy: 97.536
 [19200/60000] Loss: 0

In [15]:
import pandas as pd
import torch

# load external test.csv
test = pd.read_csv("test.csv")

# convert to tensor
X_test = torch.tensor(test.values, dtype=torch.float32).to(device)

# prediction mode
model.eval()

with torch.no_grad():
    outputs = model(X_test)
    predictions = torch.argmax(outputs, dim=1)

# cpu -> numpy
predictions = predictions.cpu().numpy()

In [16]:
submission = pd.DataFrame({
    "ImageId": range(1, len(predictions)+1),
    "Label": predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())

   ImageId  Label
0        1      2
1        2      0
2        3      9
3        4      0
4        5      3
